### Notebook 2 — Data Preparation
**Người phụ trách:** Trần Văn Khang — 52400199 — Data & Infrastructure Lead

**Bộ dữ liệu:** [Online Retail II — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

---

**Input:** `Data/Raw/online_retail_II.xlsx` 

**Output: 3 Files:**
| File | Nơi lưu | Người đọc | Vai trò |
|---|---|---|---|
| `Transactions_Clean.csv` | `Data/Processed/` | Trần Tấn Lực - 52400055 (MBA) | Giữ dòng thiếu Customer ID |
| `Customer_RFM.csv` | `Data/Processed/` | Trần Tấn Lực - 52400055 (Clustering) | RFM tính toàn kỳ |
| `Customer_Churn_Features.csv` | `Data/Processed/` | Ngô Đức Huân - 52400192 (Churn) | RFM tính cắt tại SNAPSHOT |

---

**Mục lục:**
1. Đọc dữ liệu thô và chạy Pipeline làm sạch.
2. Xuất file `Transactions_Clean.csv`.
3. Xác định mốc Snapshot cho Churn.
4. Xuất file `Customer_RFM.csv` (Clustering)
5. Xuất file `Customer_Churn_Features.csv` (Churn)
6. So sánh nhanh 2 bảng RFM

---

### 0. Chuẩn bị môi trường

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Xác định thư mục gốc project để Notebook có thể import Source/
PROJECT_NOW = Path.cwd()
if (PROJECT_NOW / "Source").exists():
    PROJECT_ROOT_TAM = PROJECT_NOW
else:
    PROJECT_ROOT_TAM = PROJECT_NOW.parent
sys.path.insert(0, str(PROJECT_ROOT_TAM))

# Import các hàm đã viết và Test kỹ ở các bước trước
from Source.Utils.IO import load_raw_excel, save_to_csv, get_project_root, PROCESSED_DATA_DIR
from Source.Preprocessing.Cleaning import clean_transactions
from Source.Preprocessing.Feature_Engineering import add_total_price_column
from Source.RFM.RFM_Calculator import calculate_rfm

print(f"Thư mục gốc dự án: {get_project_root()}")
print(f"Thư mục xuất file : {PROCESSED_DATA_DIR}")

Thư mục gốc dự án: D:\Lilith\Customer-Analytics-for-E-Commerce
Thư mục xuất file : D:\Lilith\Customer-Analytics-for-E-Commerce\Data\Processed


### 1. Đọc dữ liệu thô + Chạy Pipeline làm sạch

In [3]:
raw_data = load_raw_excel()

print(
    f"Dữ liệu thô: "
    f"{raw_data.shape[0]:,} dòng × {raw_data.shape[1]} cột\n"
)

clean_data = clean_transactions(raw_data)

Dữ liệu thô: 1,067,371 dòng × 9 cột

BẮT ĐẦU PIPELINE LÀM SẠCH DỮ LIỆU
Số dòng ban đầu: 1,067,371
----------------------------------------------------------------------
[Remove Cancelled Invoices] Đã loại 19,494 dòng Invoice huỷ (Còn lại 1,047,877 dòng).
[Remove bad debt Adjustments] Đã loại 6 dòng 'Adjust bad debt' (Còn lại 1,047,871 dòng).
[Remove Stock Adjustment_rows] Đã loại 3,457 dòng Quantity âm bất thường (Còn lại 1,044,414 dòng).
[Remove non Product Stock Codes] Đã loại 4,613 dòng StockCode phi sản phẩm (Còn lại 1,039,801 dòng).
[Remove Duplicate Rows] Đã loại 33,788 dòng trùng lặp (Còn lại 1,006,013 dòng).
----------------------------------------------------------------------
Số dòng sau khi làm sạch: 1,006,013 (đã loại tổng cộng 61,358 dòng, tương đương 5.75%)


In [4]:
# Tính thêm cột TotalPrice = Quantity * Price
# Dùng chung cho cả MBA và bắt buộc cho RFM/Churn 
df_sach = add_total_price_column(clean_data)

df_sach[["Quantity", "Price", "TotalPrice"]].describe()


,Quantity,Price,TotalPrice
count,1.006013e+06,1.006013e+06,1.006013e+06
mean,1.136363e+01,3.333701e+00,1.952892e+01
std,1.317178e+02,4.778390e+00,1.997205e+02
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.250000e+00,3.950000e+00
50%,4.000000e+00,2.100000e+00,1.000000e+01
75%,1.200000e+01,4.130000e+00,1.770000e+01
max,8.099500e+04,1.157150e+03,1.684696e+05


### 2. Xuất file `Transactions_Clean.csv`

**Quy tắc:** Giữ nguyên các dòng thiếu `Customer ID`, vì luật kết hợp (Association
Rules) chỉ cần biết "sản phẩm nào đi cùng sản phẩm nào" trong 1 hoá đơn, không cần biết ai là người mua.
Nếu loại bỏ các dòng này, MBA sẽ mất khoảng 22.8% dữ liệu giao dịch một cách không cần thiết.

In [6]:
# Không loại dòng thiếu Customer ID vì đây là input cho MBA
transactions_clean = clean_data.copy()

# Loại cột Sheet vì chỉ dùng nội bộ cho kiểm tra Duplicate
if "Sheet" in transactions_clean.columns:
    transactions_clean = transactions_clean.drop(columns=["Sheet"])

print(
    f"Transactions Clean: " 
    f"{transactions_clean.shape[0]:,} dòng × "
    f"{transactions_clean.shape[1]} cột"
)

print(
    f"Tỷ lệ vẫn còn thiếu Customer ID: "
    f"{transactions_clean['Customer ID'].isna().mean() * 100:.2f}% "
)

save_to_csv(
    transactions_clean,
    PROCESSED_DATA_DIR / "Transactions_Clean.csv"
)

Transactions Clean: 1,006,013 dòng × 8 cột
Tỷ lệ vẫn còn thiếu Customer ID: 22.80% 
Đã lưu file: D:\Lilith\Customer-Analytics-for-E-Commerce\Data\Processed\Transactions_Clean.csv  (Số dòng: 1,006,013 | Số cột: 8)


In [9]:
# Đọc lại file vừa xuất để kiểm tra format và dữ liệu
transactions_clean_check = pd.read_csv(PROCESSED_DATA_DIR / "Transactions_Clean.csv")

print(f"Shape: {transactions_clean_check.shape}")
print(f"Các cột: {list(transactions_clean_check.columns)}")

transactions_clean_check.head(5)

Shape: (1006013, 8)
Các cột: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 3. Xác định mốc Snapshot cho Churn

Trước khi xuất `Customer_RFM.csv` và `Customer_Churn_Features.csv`, cần xác định 2 mốc `snapshot_date`
khác nhau:

- **Snapshot cho RFM/Clustering:** Dùng ngày giao dịch cuối cùng trong dữ liệu (toàn kỳ).

- **Snapshot cho Churn:** Cần dựa trên phân phối *inter-purchase gap* để chọn có căn cứ thống kê, không chọn số ngẫu nhiên.

In [12]:
# Phân tích inter-purchase gap để làm căn cứ chọn Snapshot cho Churn

# Chỉ giữ các dòng đã xác định được Customer ID
customer_transaction_data = (
    clean_data.loc[
        clean_data["Customer ID"].notna()
    ]
    .sort_values(
        ["Customer ID", "InvoiceDate"]
    )
)

# Với mỗi khách hàng, lấy danh sách các thời điểm mua duy nhất (giữ nguyên cả ngày và giờ như dữ liệu gốc)
customer_purchase_times = (
    customer_transaction_data
    .groupby("Customer ID")["InvoiceDate"]
    .apply(lambda dates: sorted(dates.unique()))
)

# Tính khoảng cách giữa các lần mua liên tiếp của cùng một khách hàng
inter_purchase_gaps = []

for purchase_times in customer_purchase_times:
    if len(purchase_times) > 1:
        purchase_times = pd.to_datetime(pd.Series(purchase_times))
        gaps = (purchase_times.diff().dropna().dt.days)
        inter_purchase_gaps.extend(gaps.tolist())

inter_purchase_gap = pd.Series(inter_purchase_gaps)

# Số khách có từ 2 thời điểm mua khác nhau trở lên
customers_with_multiple_purchases = (customer_purchase_times.apply(len).gt(1).sum())

# Số khách chỉ mua đúng 1 lần
customers_with_one_purchase = (customer_purchase_times.apply(len).eq(1).sum())

print(
    f"Số cặp lần mua liên tiếp thu được: "
    f"{len(inter_purchase_gap):,} "
    f"(từ {customers_with_multiple_purchases:,} "
    f"khách có ≥ 2 lần mua khác nhau)"
)

print(
    f"\nSố khách chỉ mua đúng 1 lần: "
    f"{customers_with_one_purchase:,} "
    f"/ {len(customer_purchase_times):,} "
    f"({customers_with_one_purchase / len(customer_purchase_times) * 100:.1f}%) "
)

print(
    "\nPhân phối Inter Purchase Gap (ngày) theo Percentile: "
)

for percentile in [50, 75, 80, 90, 95, 99]:
    print(
        f"  P{percentile}: "
        f"{inter_purchase_gap.quantile(percentile / 100):.1f} ngày"
    )

Số cặp lần mua liên tiếp thu được: 30,632 (từ 4,235 khách có ≥ 2 lần mua khác nhau)

Số khách chỉ mua đúng 1 lần: 1,618 / 5,853 (27.6%) 

Phân phối Inter Purchase Gap (ngày) theo Percentile: 
  P50: 25.0 ngày
  P75: 62.0 ngày
  P80: 77.0 ngày
  P90: 136.0 ngày
  P95: 207.0 ngày
  P99: 369.0 ngày


**Nhận xét:** 
- Phân phối khoảng cách giữa các lần mua liên tiếp có **trung vị 25 ngày**, nhưng có đuôi phải dài: **P75 = 62 ngày, P90 = 136 ngày, P95 = 207 ngày và P99 = 369 ngày**. 

- Điều này cho thấy hành vi tái mua giữa các khách hàng có mức độ phân tán đáng kể, với một số khách có khoảng cách giữa các lần mua lên tới hàng trăm ngày. 

- Các Percentile này cung cấp cơ sở thống kê để xem xét và lựa chọn cửa sổ quan sát nhãn churn phù hợp. **Quyết định Snapshot, Churn Window và lý giải chính thức** trong `Source/Churn/Label_Generator.py`.

In [16]:
# Xác định Snapshot cho Clustering và Churn SNAPSHOT_CHURN hiện chỉ là Placeholder

CHURN_WINDOW_DAYS_PLACEHOLDER = int(inter_purchase_gap.quantile(0.75))

print(
    f"Độ dài cửa sổ churn Placeholder (Percentile 75): "
    f"{CHURN_WINDOW_DAYS_PLACEHOLDER} ngày"
)

# Ngày giao dịch cuối cùng trong dữ liệu đã làm sạch
last_transaction_date = clean_data["InvoiceDate"].max()

# Snapshot toàn kỳ
SNAPSHOT_TOAN_KY = last_transaction_date

# Snapshot cắt sớm
SNAPSHOT_CHURN = (last_transaction_date - pd.Timedelta(days=CHURN_WINDOW_DAYS_PLACEHOLDER))

print(
    f"\nSnapshot toàn kỳ: "
    f"{SNAPSHOT_TOAN_KY}"
)

print(
    f"Snapshot Churn: "
    f"{SNAPSHOT_CHURN} "
)

Độ dài cửa sổ churn Placeholder (Percentile 75): 62 ngày

Snapshot toàn kỳ: 2011-12-09 12:50:00
Snapshot Churn: 2011-10-08 12:50:00 


### 4. Xuất file `Customer_RFM.csv` (Clustering)

Gọi hàm `calculate_rfm()` với `snapshot_date = SNAPSHOT_TOAN_KY` — Tức dùng toàn bộ lịch sử giao dịch để mô tả khách hàng. Đây là Input cho bước phân khúc khách hàng (Customer Segmentation), không có khái niệm "dự đoán tương lai" nên dùng toàn bộ dữ liệu là hợp lệ.

In [19]:
# Tính RFM toàn kỳ cho Customer Segmentation

# Đảm bảo clean_data có cột TotalPrice trước khi tính Monetary
if "TotalPrice" not in clean_data.columns:
    clean_data = add_total_price_column(clean_data)

customer_rfm = calculate_rfm(clean_data, snapshot_date=SNAPSHOT_TOAN_KY)

print("\nThống kê RFM (toàn kỳ): ")

customer_rfm.describe()

[Calculate RFM] Đã loại 229,371 dòng thiếu 'Customer ID' (Còn lại 776,642 dòng để tính RFM).
[Calculate RFM] Mốc Snapshot Date = 2011-12-09. Đã loại 0 dòng có ngày giao dịch Sau mốc này (Còn lại 776,642 dòng).
  (Lưu ý: 0 dòng bị loại nghĩa là Snapshot Date đang lớn hơn hoặc bằng ngày giao dịch cuối cùng trong dữ liệu.
[Calculate RFM] Đã tính RFM cho 5,853 khách hàng.

Thống kê RFM (toàn kỳ): 


,Customer ID,Recency,Frequency,Monetary
count,5853.000000,5853.000000,5853.000000,5853.000000
mean,15319.354519,199.166240,6.253203,2916.335857
std,1714.995565,208.505959,12.751977,14305.788772
min,12346.000000,0.000000,1.000000,0.000000
25%,13837.000000,24.000000,1.000000,339.500000
50%,15320.000000,94.000000,3.000000,856.010000
75%,16802.000000,378.000000,7.000000,2240.900000
max,18287.000000,738.000000,373.000000,580987.040000


In [25]:
# Xuất file Customer_RFM.csv

save_to_csv(customer_rfm, PROCESSED_DATA_DIR / "Customer_RFM.csv")

customer_rfm_check = pd.read_csv(PROCESSED_DATA_DIR / "Customer_RFM.csv")

print(f"Shape: {customer_rfm_check.shape}")
print(f"Các cột: {list(customer_rfm_check.columns)} ")

customer_rfm_check.head(5)

Đã lưu file: D:\Lilith\Customer-Analytics-for-E-Commerce\Data\Processed\Customer_RFM.csv  (Số dòng: 5,853 | Số cột: 4)
Shape: (5853, 4)
Các cột: ['Customer ID', 'Recency', 'Frequency', 'Monetary'] 


,Customer ID,Recency,Frequency,Monetary
0,12346.0,325,3,77352.96
1,12347.0,1,8,4921.53
2,12348.0,74,5,1658.40
3,12349.0,18,3,3678.69
4,12350.0,309,1,294.40


### 5. Xuất file `Customer_Churn_Features.csv` (Churn)

Đây là điểm kỹ thuật quan trọng nhất để chống rò rỉ dữ liệu: Các giao dịch xảy ra sau mốc `SNAPSHOT_CHURN` sẽ không được dùng để tính RFM ở đây để xác nhận khách có thực sự quay lại mua hay không.

In [27]:
# Tính RFM cắt tại Snapshot Churn

# Đảm bảo có TotalPrice trước khi tính Monetary
if "TotalPrice" not in clean_data.columns:
    clean_data = add_total_price_column(clean_data)

customer_churn_features = calculate_rfm(clean_data, snapshot_date=SNAPSHOT_CHURN)

print(
    "\nThống kê RFM "
    " (Cắt tại Snapshot Churn - Placeholder): "
)

customer_churn_features.describe()

[Calculate RFM] Đã loại 229,371 dòng thiếu 'Customer ID' (Còn lại 776,642 dòng để tính RFM).
[Calculate RFM] Mốc Snapshot Date = 2011-10-08. Đã loại 116,778 dòng có ngày giao dịch Sau mốc này (Còn lại 659,864 dòng).
[Calculate RFM] Đã tính RFM cho 5,472 khách hàng.

Thống kê RFM  (Cắt tại Snapshot Churn - Placeholder): 


,Customer ID,Recency,Frequency,Monetary
count,5472.000000,5472.000000,5472.000000,5472.000000
mean,15318.675621,203.426718,5.800804,2687.816343
std,1710.655732,186.697810,11.592128,12993.814994
min,12346.000000,0.000000,1.000000,2.900000
25%,13838.750000,32.000000,1.000000,321.125000
50%,15313.500000,144.000000,3.000000,785.030000
75%,16798.250000,339.000000,6.000000,2112.040000
max,18287.000000,676.000000,308.000000,524476.600000


In [29]:
save_to_csv(customer_churn_features, PROCESSED_DATA_DIR / "Customer_Churn_Features.csv")

customer_churn_features_check = pd.read_csv(PROCESSED_DATA_DIR / "Customer_Churn_Features.csv")

print(f"Shape: {customer_churn_features_check.shape}")
print(
    f"Các cột: "
    f"{list(customer_churn_features_check.columns)}"
)

customer_churn_features_check.head(5)

Đã lưu file: D:\Lilith\Customer-Analytics-for-E-Commerce\Data\Processed\Customer_Churn_Features.csv  (Số dòng: 5,472 | Số cột: 4)
Shape: (5472, 4)
Các cột: ['Customer ID', 'Recency', 'Frequency', 'Monetary']


,Customer ID,Recency,Frequency,Monetary
0,12346.0,263,3,77352.96
1,12347.0,67,6,3402.39
2,12348.0,12,5,1658.40
3,12349.0,345,2,2221.14
4,12350.0,247,1,294.40


### 6. So sánh nhanh 2 bảng RFM

In [32]:
rfm_snapshot_comparison = pd.DataFrame({
    "Chỉ số": [
        "Số khách hàng",
        "Frequency trung bình",
        "Monetary trung bình (£)",
        "Recency trung bình (ngày)",
    ],
    
    "Customer_RFM.csv (Toàn kỳ)": [
        len(customer_rfm),
        round(customer_rfm["Frequency"].mean(), 2),
        round(customer_rfm["Monetary"].mean(), 2),
        round(customer_rfm["Recency"].mean(), 2),
    ],
    
    "Customer_Churn_Features.csv (Cắt Snapshot)": [
        len(customer_churn_features),
        round(customer_churn_features["Frequency"].mean(), 2),
        round(customer_churn_features["Monetary"].mean(), 2),
        round(customer_churn_features["Recency"].mean(), 2),
    ],
})

rfm_snapshot_comparison

,Chỉ số,Customer_RFM.csv (Toàn kỳ),Customer_Churn_Features.csv (Cắt Snapshot)
0,Số khách hàng,5853.00,5472.00
1,Frequency trung bình,6.25,5.80
2,Monetary trung bình (£),2916.34,2687.82
3,Recency trung bình (ngày),199.17,203.43


**Nhận xét:** 
- Bảng trên cho thấy rõ sự khác biệt giữa hai bộ RFM. So với bản toàn kỳ, bản cắt tại `SNAPSHOT_CHURN` có **Frequency trung bình thấp hơn (5.80 so với 6.25)** và **Monetary trung bình thấp hơn (£2,687.82 so với £2,916.34)**, đồng thời số khách hàng giảm từ **5,853 xuống 5,472** do chỉ những khách hàng có giao dịch trước hoặc tại Snapshot mới được đưa vào Feature. 

- Sự khác biệt này minh họa trực tiếp tác động của việc cắt dữ liệu theo thời gian và là cơ sở để tránh sử dụng `Customer_RFM.csv` (toàn kỳ) làm Feature cho mô hình Churn, vì khi đó các giao dịch xảy ra sau thời điểm dự đoán có thể được đưa vào Feature, gây **rò rỉ dữ liệu**.
